In [0]:
%pip install kafka-python

In [0]:
import tempfile, ssl, json
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

#Get secrets

In [0]:
host = dbutils.secrets.get("jdbc", "kafka-host")
port = dbutils.secrets.get("jdbc", "kafka-port")
topic = dbutils.secrets.get("jdbc", "kafka-topic")

#Write certs to temp files

In [0]:
tmpdir = tempfile.mkdtemp()
with open(f"{tmpdir}/service.key", "w") as f:
    f.write(dbutils.secrets.get("jdbc", "kafka-access-key"))

with open(f"{tmpdir}/service.cert", "w") as f:
    f.write(dbutils.secrets.get("jdbc", "kafka-access-cert"))

with open(f"{tmpdir}/ca.pem", "w") as f:
    f.write(dbutils.secrets.get("jdbc", "kafka-ca-cert"))

#Define schema for incoming events 

In [0]:
schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("product_key", StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("event_time",  StringType())
])

#Read from Kafka

In [0]:
df_raw = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", f"{host}:{port}")
         .option("subscribe", topic)
         .option("startingOffsets", "earliest")
         .option("kafka.security.protocol", "SSL")
         .option("kafka.ssl.truststore.type", "PEM")
         .option("kafka.ssl.keystore.type", "PEM")
         .option("kafka.ssl.truststore.certificates", dbutils.secrets.get("jdbc", "kafka-ca-cert"))
         .option("kafka.ssl.keystore.certificate.chain", dbutils.secrets.get("jdbc", "kafka-access-cert"))
         .option("kafka.ssl.keystore.key", dbutils.secrets.get("jdbc", "kafka-access-key"))
         .load()
)

#Parse JSON messages 

In [0]:
df_parsed = (
    df_raw
    .select(from_json(col("value").cast("string"), schema).alias("data"),
            col("timestamp").alias("kafka_timestamp"))
    .select("data.*", "kafka_timestamp")
    .withColumn("ingested_at", current_timestamp())
)

#Write to Bronze Delta Table 

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.checkpoints")

checkpoint = "/Volumes/workspace/bronze/checkpoints/kafka_sales_events"
query = (
    df_parsed.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", checkpoint)
             .trigger(availableNow=True)
             .toTable("workspace.bronze.kafka_sales_events")
)

query.awaitTermination()
print("Streaming complete — check workspace.bronze.kafka_sales_events")

#Check Data

In [0]:
df = spark.table("workspace.bronze.kafka_sales_events")
print(f"Row count: {df.count()}")
df.display()